Define the list of "mother" supernovae that are to be used as basis for WarpTemplates.

- Look at things class by class.
- Define subset of BTS supernovae below some redshift cat, with type and minimal data quality.
- For each SN, look at the template fits and identify a subset of templates used as warp engines.

These cuts can be redone later based on subsequent analysis steps.

Things we wish to determine:
- How many classes should we consider?
- How to select how many templates to retain, and whether these should have weights?
- Determine a max redshift, and whether this should vary per class?
- Lightcurve quality criteria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
import json
from astropy.cosmology import Planck13 as cosmo

Lessons learned:

From first run through, examination of the original bts classes. Which to determine which can be used, which should be added to others and what redshift limit to use.
- Ca-rich: Single, skip class.
- ILRT: Only two, skip class.
- SLSN-II: zrange 0.07, 0.17
- SN Ia-91 bg: default z ok (could increase to 0.05 to add some more, but biased)
- SN Ia-91T: z range 0.03 < 0.08,
- SN Ia-CSM: skip, fluctuations anyway at late phases.
- SN Ia-pec: <0.06 (0.055), combine with other weird?
- SN Iax: <0.055. Keep alone or add to Ia-pec?
- SN Ib: Default is fine.
- SN Ibn: Combine with Ib?
- SN Ic: Default is fine.
- SN Ic-BL: Combine with Ic.
- SN Icn: Single, skip class.
- SN II: Default is fine (could increase range?)
- SN II-pec: add to SNII.
- SN IIL: add to SN II
- SN IIP: Keep as is, but could also include in general SNII
- SN Ib/c: Keep range, but then also add Ib/Ic for parent class?
- TDE: Could work, but better models exist. So skip for now.

  Some files not found, like SN IIn - why?
  What about the SNIa classes? (Iax, 91bg, 91T ? Should these start from salt templates or other keep now?)

In [ ]:
# References warpclasses 
classnbr = 0

In [ ]:
# This is the classes which we eventually expect will be made into 
# warptemplate classes. 
# Todo: some BTS classes mising, like SLSN-I & SN IIn. 
#       Probably for some good reason, but need to verify for paper.
warpclasses = [
    'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T',
    'SN Ib/c', 'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic',
    'SLSN-II']
# Some of these will be umbrella classes and we will combine btsfits files:
parentclasses = {
    'SN Ic':['SN Ic', 'SN Ic-BL', 'SN Ic-pec', 'SN Icn'], 
    # Note that "/" not used in filenames
    'SN Ib/c':['SN Ibc', 'SN Ib', 'SN Ic', 'SN Ibn', 'SN Ic-BL', 'SN Ic-pec', 'SN Icn'], 
    'SN II': ['SN II','SN IIP', 'SN II-pec', 'SN IIL']
}
# Redshift ranges for particular classes
zclass = {
    'SLSN-II': [0.07, 0.17],
    'SN Ia-91bg': [0.01, 0.055],
    'SN Ia-91T': [0.03, 0.08],
    'SN Ia-pec': [0.01, 0.055], 
    'SN Iax': [0.01, 0.055], 
}

In [ ]:
# So, what are our recommendeded limits?
# These are the things we can try
# Ok, fix these ... seems like we cannot really increase the requirements 
# and keep sufficient data. Instead, think of a ranking of data
# and use the best fits.
zlim = [0.01,0.04]
min_bands= 2
min_peakndof = 6
require_peak_good = False
# Fixed params
fdir = '/Users/jnordin/data/models/sncosmo/'
peakfit = True          # Only use data round peak to estimate fit quality 
aiceval = False        # Rank fits based on aic (True) or chi/dof (False)
# Settings for template running - optimized previoiusly
phaserange = 60
intdisp =  0.1
fracdisp = 0
peak_chicut = 1.5

In [ ]:
if warpclasses[classnbr] in zclass:
    zlim = zclass[warpclasses[classnbr]]

### Define true match
We need a mapping to check whether a given SNcosmo class should be considered a true fit to a given BTS def.

In [ ]:
# For each warpclass, define which sncosmo templates to be 
# considered "true", if only including such templates as models.
bts_truth_map = {
    'SN Ib': ['SN Ic', 'SN Ib/c', 'SN Ib', 'SN Ic-BL'], 
    'SN Ia-91bg': ['SN Ia'], 
    'SN II': ['SN II', 'SN IIL', 'SN IIl', 'SN IIp', 'SN IIB','SN IIb', 'SN IIP', 'SN IIn', 'SN IIN'], 
    'SN Ia-91T': ['SN Ia'], 
    'SN Ib/c': ['SN Ic', 'SN Ib/c', 'SN Ib', 'SN Ic-BL'], 
    'SN Ia-pec': ['SN Ia'], 
    'SN IIP': ['SN II', 'SN IIp' 'SN IIP'], 
    'SN Iax': ['SN Ia'], 
    'SN Ic': ['SN Ic', 'SN Ib/c', 'SN Ib', 'SN Ic-BL'],
    'SLSN-II': ['SN II', 'SN IIL', 'SN IIl', 'SN IIp', 'SN IIB','SN IIb', 'SN IIP', 'SN IIn', 'SN IIN', 'PopIII'],     # No templates available, using these for now
    'SN IIn': ['SN II', 'SN IIL', 'SN IIl', 'SN IIp', 'SN IIB','SN IIb', 'SN IIP', 'SN IIn', 'SN IIN'], 
}

### Start processing

In [ ]:
def combine_typefits( dictlI, dictlII ):
    """
    Combine the internal dicts of two lists of dicts.
    """
    for templatename, datalist in dictlII.items():
        dictlI[templatename].extend( datalist )
    return dictlI 

In [ ]:
typefitdata = None
if warpclasses[classnbr] in parentclasses:
    toread = parentclasses[warpclasses[classnbr]]
else:
    toread = [warpclasses[classnbr]]

for readclass in toread:
    rootname = 'btsfits_{}_{}_{}_{}_{}'.format(
        readclass, str(phaserange), str(peak_chicut), str(fracdisp), str(intdisp)
    )
    fname = fdir+rootname+'.json'
    # Open data
    with open(fname, "r") as infile:
        if typefitdata is None:
            typefitdata = json.loads(infile.read())
        else:
            typefitdata = combine_typefits( 
                typefitdata, json.loads(infile.read()) 
            )
    
    print(readclass, len(typefitdata))

In [ ]:
def get_salt_cosmofit( fitlist,
                        chidofmax=3,    # Max chi/dof to be counted as good fit
                        truetypes = []
                     ):
    """
    Go through the input list of saltfits. Return a list of the subset which fulfills
    selection criteria as specified. The idea is that the provided list of templates
    are "good enough" for template construction.

    Parameters:
    - 
    determine whether the fit is good and whether it 
    could be considered for cosmology.

    If the model class included in truetypes, classification set to correct.

    Nothing returned for failed fits.

    Return a pandas df
    """

    outd = []

    for snfit in fitlist:
        if not snfit['success']:
            continue
        snd = {
            k:snfit[k] for k in ['z', 'chisq', 'ndof', 'peakmag', 'chidof', 'id', 'nbr_bands', 'ndet','class',
                                 'peakchi','peakdet','earlydet','peakbands','peak_good']
        }
        fitp = snfit['ndet']-snfit['ndof']
        snd['aic'] = 2 * fitp + snd['chisq'] # 4 dof, ln L = - 0.5 chi 
        snd['peakndof'] = snfit['peakdet']-fitp

        if peakfit:
            if not snd['peakndof']>0:
                continue
            snd['peakchisqdof'] = snfit['peakchi']/snd['peakndof']
            snd['peakaic'] = 2 * fitp + snfit['peakchi'] # 4 dof, ln L = - 0.5 chi
            snd['chidof'] = snfit['peakchi']/snd['peakndof']
            snd['aic'] = snd['peakaic']
        else:
            snd['chidof'] = snfit['chidof']

            
        # Check for good, correct fit
        snd['correct'] = False
        if snd['class'] in truetypes:
            snd['correct'] = True
        #snd['goodfit'] = False
        #if ( (-4 < snfit['x1'] < 4) and (-0.5 < snfit['c'] < 1.5) and (snd['chidof'] < chidofmax) 
        #    and (snfit['errors']['x1'] < 2.) and (snfit['errors']['c'] < 1.) ):
        #    snd['goodfit'] = True
        if (snd['chidof'] < chidofmax):
            snd['goodfit'] = True
        else:
            snd['goodfit'] = False

        outd.append(snd)

    return pd.DataFrame.from_dict(outd)


def get_timeseries_goodfit( fitlist,
                        chidofmax=3,    # Max chi/dof to be counted as good fit
                        truetypes = [],                           
                          ):
    """
    Go through the input list of an sncosmo timeseries model with added dust. 
    For each, determine whether the fit is good.

    Nothing returned for failed fits.

    Assumes fit made with amplitude, rv and ebv. Should probably redo without fitting both ... 

    Return a pandas df
    """

    outd = []

    for snfit in fitlist:
        if not snfit['success']:
            continue
        snd = {
            k:snfit[k] for k in ['z', 'chisq', 'ndof', 'peakmag', 'chidof', 'id', 'nbr_bands', 'ndet','class',
                                 'peakchi','peakdet','earlydet','peakbands','peak_good']
                                 
        }
        fitp = snfit['ndet']-snfit['ndof']
        snd['aic'] = 2 * fitp + snd['chisq'] # ln L = - 0.5 chi 

        snd['peakndof'] = snfit['peakdet']-fitp

        if peakfit:
            if not snd['peakndof']>0:
                continue
            snd['peakchisqdof'] = snfit['peakchi']/snd['peakndof']
            snd['peakaic'] = 2 * fitp + snfit['peakchi'] # 4 dof, ln L = - 0.5 chi
            snd['chidof'] = snfit['peakchi']/snd['peakndof']
            snd['aic'] = snd['peakaic']
        else:
            snd['chidof'] = snfit['chidof']


        
        # Check for good fit (note that r_v is typically not find and should be trivially fulfilled)
        #if ( ( -0.3 < snfit['hostebv'] < 2.5) and (0.1 < snfit['hostr_v'] < 5) and (snd['chidof'] < chidofmax) ):
        #    snd['goodfit'] = True
        #else:
        #    snd['goodfit'] = False

        if (snd['chidof'] < chidofmax):
            snd['goodfit'] = True
        else:
            snd['goodfit'] = False

            
        if snd['class'] in truetypes:
            snd['correct'] = True
        else:
            snd['correct'] = False

        outd.append(snd)

    return pd.DataFrame.from_dict(outd)


In [ ]:
modelfits = {}
for modelname, modeldata in typefitdata.items():
    if re.search('salt', modelname):
        df = get_salt_cosmofit( modeldata, truetypes=bts_truth_map[warpclasses[classnbr]] )
    else:
        df = get_timeseries_goodfit( modeldata, truetypes=bts_truth_map[warpclasses[classnbr]] )
    df.insert(0,'model', modelname) 
    
    modelfits[modelname] = df

In [ ]:
# Combine to one df
dfall = pd.concat( modelfits.values(), ignore_index=True )

In [ ]:
dfall

In [ ]:
def parse_fits( fitdf, suffix='', aiceval=True, allgood_count=3):
    """
    Parse a dataframe summarizing fits to a SN. 
    
    This is what we have to redo ... 
    
    Will provide:
    - best template name, class, chi, par, ndet, correct, good
    - nbr of templates similar to best match and types contained there. 

    Whether all templates should be considered good is either through aic matching (aiceval=True) or 
    checking whether the `allgood_count` best all have correct class (if aiceval=False)
    
    Akaike weights (basically likelihood ratios):
    https://sites.warnercnr.colostate.edu/wp-content/uploads/sites/73/2017/05/Burnham-and-Anderson-2004-SMR.pdf
    """

    if len(fitdf)==0:
        return {}

    
    best = fitdf.iloc[0,:]
    info = {
        k+suffix:best[k] for k in 
        ['model', 'chisq','chidof','nbr_bands','class','aic','goodfit','cosmofit','correct','lowdust','peak_good']
    }

    # Get all similarly good fits
    deltaaic = fitdf['aic']-best['aic']
    aicw = np.exp(- deltaaic/2) / np.sum(np.exp(- deltaaic/2))
    matchdf = fitdf[aicw>0.001]
    info['aicmatches_nbr'+suffix] = matchdf.shape[0]
    info['aicmatches_types'+suffix] = '_'.join( set( matchdf['class'] ) )
    info['aicmatches_allgood'+suffix] = matchdf['correct'].all() 

    if aiceval:
        info['matches_nbr'+suffix] = matchdf.shape[0]
        info['matches_types'+suffix] = '_'.join( set( matchdf['class'] ) )
        info['matches_allgood'+suffix] = matchdf['correct'].all()
    else:
        # Look at the best matches 
        matchdf = fitdf.iloc[0:allgood_count,:]
        info['matches_nbr'+suffix] = matchdf.shape[0]
        info['matches_types'+suffix] = '_'.join( set( matchdf['class'] ) )
        info['matches_allgood'+suffix] = matchdf['correct'].all()
        # Comparison
        matchdf = fitdf.iloc[-allgood_count:,:]        

    return info

    

In [ ]:
# What do we want to do here?

snstore = []
for name, sndf in dfall.sort_values(['id','chidof'], ascending=True).groupby('id'):
#    print(name)
    # Now we can inspect and evaluate this sn ... 
    snsum = {
        'id':name, 'class':warpclasses[classnbr], 'z':sndf['z'].iloc[0], 'peakmag':sndf['peakmag'].iloc[0],
        'ndet':sndf['ndet'].iloc[0], 'nbr_bands':sndf['nbr_bands'].iloc[0], 
        'peakndof':sndf['peakndof'].iloc[0], 'peak_good':sndf['peak_good'].iloc[0], 
    }

    snstore.append( snsum )    
    

In [ ]:
dfsum = pd.DataFrame.from_dict( snstore )

In [ ]:
dfsum['absmag'] = dfsum['peakmag'] - cosmo.distmod(dfsum['z']).value

In [ ]:
if require_peak_good:
    iCut = ( 
        (dfsum['z']>zlim[0]) & (dfsum['z']<zlim[1]) & 
        (dfsum['nbr_bands']>=min_bands) & (dfsum['peakndof']>=min_peakndof)
        & dfsum['peak_good']
    )
else:
    iCut = ( 
        (dfsum['z']>zlim[0]) & (dfsum['z']<zlim[1]) & 
        (dfsum['nbr_bands']>=min_bands) & (dfsum['peakndof']>=min_peakndof)
    )


In [ ]:
print('{} out of {} surviving'.format(sum(iCut),len(snstore)))

In [ ]:
plt.figure(1, figsize=(10,3))
plt.subplot(1,5,1)
plt.plot(dfsum['z'], dfsum['peakmag'],'o')
plt.plot(dfsum['z'][iCut], dfsum['peakmag'][iCut],'o')
plt.subplot(1,5,2)
plt.plot(dfsum['z'], dfsum['ndet'],'o')
plt.plot(dfsum['z'][iCut], dfsum['ndet'][iCut],'o')
plt.subplot(1,5,3)
plt.plot(dfsum['z'], dfsum['peakndof'],'o')
plt.plot(dfsum['z'][iCut], dfsum['peakndof'][iCut],'o')
plt.subplot(1,5,4)
plt.plot(dfsum['z'], dfsum['nbr_bands'],'o')
plt.plot(dfsum['z'][iCut], dfsum['nbr_bands'][iCut],'o')
plt.subplot(1,5,5)
plt.plot(dfsum['z'], dfsum['absmag'],'o')
plt.plot(dfsum['z'][iCut], dfsum['absmag'][iCut],'o')


In [ ]:
labelpanel = True
cm = sns.color_palette("husl", 5)
cm = sns.color_palette("husl",2)

In [ ]:
plt.figure(1, figsize=(4,4))
plt.title(warpclasses[classnbr])

# Mag vs z 
#plt.subplot(1,4,1)
if labelpanel:
    plt.plot(dfsum['z'], dfsum['absmag'],'o', color=cm[1], 
            label='All ({})'.format(len(snstore)))
    plt.plot(dfsum['z'][iCut], dfsum['absmag'][iCut],'o',color=cm[0], 
             markeredgecolor='black',  
            label='Warp seed ({})'.format(sum(iCut)))
else:
    plt.plot(dfsum['z'], dfsum['absmag'],'o', color=cm[1])
    plt.plot(dfsum['z'][iCut], dfsum['absmag'][iCut],'o',color=cm[0], markeredgecolor='black')
    plt.xlabel('Redshift (z)')
plt.ylabel('Peak abs mag (ZTF g/R)')
plt.legend()

# det vs z 
#plt.subplot(1,4,2)
#_, bins, __ = plt.hist(dfsum['peakndof'], bins=20, color=cm[1])
#_, bins, __ = plt.hist(dfsum['peakndof'][iCut], bins=bins, color=cm[0])
#plt.xlabel('Redshift (z)')
#plt.xlabel('Peak detections')
#plt.legend()


plt.tight_layout()
plt.savefig( fdir+'warpmod/'+re.sub(r'/', '', warpclasses[classnbr])+'.png', dpi=300 )

Now we have a base series of classes. Now we wish to determine parameters. The goal is to make sure that we have a representative sample for each class (with good lightcurves) and that each of them has a "good" sample of supernova templates to start from. 
How do we study this?
- Numbers of vs redshift?
- chi/dof or aic, nbr of templates per sn etc.
- k

So, decide on some set of parameters, run them through for all of the classes, then choose.
Three sample for each: all of BTS, good fits, good fits in final selection.
We plot
- peak mag vs redshift
- chidof vs redshift
- aic vs redshift
- nbr of templates per SN (hist) 

### Results of investigations to requirements for warp seeds
Main goal is to find whether a reasonable number of templats remain.
Want to have as high req as possible while keeping diversity

zlim14_bands2_dof6_allpeak
- sufficient count for all, so maybe go with this? 

### Create output table for this warpclass.
After each warp sn, add a list of templates and their relative probabilities

In [ ]:
tabI = []
tabII = []
for k, row in dfsum[iCut].iterrows():
    print(row['id'])
    tabI.append( row[['id','z','peakmag', 'ndet', 'nbr_bands']] )

    # Subset of template fits of correct "type" (however defined) for this sn
    sninfo = dfall[ (dfall['id']==row['id']) & (dfall['correct']==True) ]
    keep = ['id','model', 'class', 'earlydet', 
       'peak_good', 'peakndof', 'peakchisqdof', 'goodfit']
    tabII.append( sninfo[keep] )
    continue
    foo = pd.merge( [foo, np.array(bla)], ignore_index=True, sort=False)
    print(foo)
    continue
    print(bla['id'])
    print(pd.merge(bla,foo, suffixes=('', '_template')))
    break
    
    foo2 = foo.sort_values(['peakchisqdof'], ascending=True)
    if foo2.shape[0] > maxtemplates:
        foo2 = foo2.iloc[0:maxtemplates,:]

    foo2.loc[:,'peakchi'] =  foo2['peakchisqdof'] * foo2['peakndof']
    foo2.loc[:,'prop'] =  np.exp( -foo2['peakchisqdof'] * foo2['peakndof'] )
    # 1. Use peakndof  peakchisqdof      peakaic to determine relative probability
    # 2. For each template, store ID, nbr_bands, class, peak_good
    # In next notebook
    # 3. For each sn, plot the template fits up to a certain max entry / min prob and highlight peak_good
    print(foo2)


In [ ]:
dfI = pd.DataFrame.from_dict( tabI )

In [ ]:
dfII = pd.concat( tabII )

In [ ]:
dftot = pd.merge(dfI, dfII )

In [ ]:
dftot.to_csv( fdir+'warpmod/templatematches_'+re.sub(r'/', '', warpclasses[classnbr])+'.csv' )